### 데이터의 분할

- 분석 모델을 학습하고 성과를 확인하기 위해서 데이터를 Train, Test 세트로 나눠주는 작업
- 독립 변수와 종속 변수 모두 Train, Test 데이터로 나눠준다.
- 데이터를 분할하는 일반적인 비율
    - Train : Test = 7:3 // 8:2
    - Train : Validation : Test = 6:2:2
        - 데이터의 양이 많은 경우 검증 셋을 생성하여 2번의 검증 작업
    - k-fold 교차검증, 장점: 신뢰도 상승, 단점: 오래 걸림
    - 유의할 점
        - 데이터의 불균형 문제
        - 데이터의 균형을 알맞게 분할하는 작업이 필요
- sklearn 안에 train_test_split() 함수를 이용하여 데이터를 분할
    - train_test_split(X, y, test_size = None, random_state = None, shuffle = True, stratify = None)
        - X: 독립 변수(문제)
        - y: 종속 변수(정답)
        - test_size: test 데이터의 비율 (0~1)
        - random_state: 임의의 데이터를 추출하는 과정에서 seed 값을 지정
        - shuffle: 데이터를 섞을 것인가(시계열 데이터셋인 경우에는 False)
        - stratify: 특정 변수를 지정하면 해당 변수를 기준으로 계층화. 해당 변수의 비율을 유지하도록 데이터 분할

In [1]:
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

In [2]:
iris_data = load_iris()

In [3]:
iris = pd.DataFrame(iris_data['data'], columns = iris_data['feature_names'])

iris['class'] = iris_data['target']

iris.head()

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),class
0,5.1,3.5,1.4,0.2,0
1,4.9,3.0,1.4,0.2,0
2,4.7,3.2,1.3,0.2,0
3,4.6,3.1,1.5,0.2,0
4,5.0,3.6,1.4,0.2,0


In [4]:
# target 데이터들을 변환

iris['class'] = iris['class'].map(
    lambda x: iris_data['target_names'][x]
)

In [5]:
iris

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),class
0,5.1,3.5,1.4,0.2,setosa
1,4.9,3.0,1.4,0.2,setosa
2,4.7,3.2,1.3,0.2,setosa
3,4.6,3.1,1.5,0.2,setosa
4,5.0,3.6,1.4,0.2,setosa
...,...,...,...,...,...
145,6.7,3.0,5.2,2.3,virginica
146,6.3,2.5,5.0,1.9,virginica
147,6.5,3.0,5.2,2.0,virginica
148,6.2,3.4,5.4,2.3,virginica


In [6]:
iris['class'].value_counts()

class
setosa        50
versicolor    50
virginica     50
Name: count, dtype: int64

In [7]:
X = iris.drop('class', axis=1)
y = iris['class']

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size = 0.3, random_state = 42
)

In [9]:
# 데이터의 개수를 확인
print(f'X_train: {X_train.shape}, X_test: {X_test.shape}')
print(f'y_train: {y_train.shape}, y_test: {y_test.shape}')

X_train: (105, 4), X_test: (45, 4)
y_train: (105,), y_test: (45,)


In [10]:
y_train.value_counts()

class
versicolor    37
virginica     37
setosa        31
Name: count, dtype: int64

In [11]:
X_train2, X_test2, y_train2, y_test2 = train_test_split(
    X, y, test_size = 0.3, shuffle=False
)

In [12]:
y_train2.value_counts()

class
setosa        50
versicolor    50
virginica      5
Name: count, dtype: int64

In [17]:
X_train3, X_test3, y_train3, y_test3 = train_test_split(
    X, y, test_size = 0.3, stratify=y, random_state=42
)

In [18]:
y_train3.value_counts()

class
versicolor    35
setosa        35
virginica     35
Name: count, dtype: int64

### 데이터 스케일링
- 대부분의 분석 알고리즘은 컬럼 간 데이터의 범위가 크게 차이나는 경우에 정상적으로 작동하지 않는다. (성능이 떨어진다.)
- 값의 범위가 작은 컬럼에 비해서 범위가 큰 컬럼이 종속 변수를 예측하는데 큰 영향을 준다고 판단
- 따라서 스케일링 작업은 모든 컬럼의 값의 범위를 같게 만들어주는 작업
- 스케일링 작업 전에 이상치를 대체하거나 제거하여야 한다.
- 스케일링 작업 순서
    - train, test로 나눠져있는 경우 train에서 사용한 scaler를 test에서 사용(일반적인 방법)
    1. Scaler 선택
        - 해당 Scalr를 import
    2. Scaler 객체(class) 생성
    3. train 데이터의 분포(범위)를 저장 (fit())
    4. train 데이터를 스케일링 (transform())
    5. test 데이터를 스케일링 (transform())
    6. 원본 데이터로 변환 (inverse_transform())

#### Standard Scaler
- 표준화 방식: 가장 기본적인 스케일러
- 평균이 0, 분산이 1인 정규분포로 스케일링
- 최솟값과 최댓값의 크기를 따로 제한하지 않아 이상치에 민감: 이상치에 대한 확인 및 정제를 한 뒤 사용
- 일반적으로 회귀분석보다는 분류분석에서 사용

In [16]:
from sklearn.preprocessing import StandardScaler

In [19]:
# class 생성

stdscaler = StandardScaler()

In [20]:
# X_train3 데이터의 범위를 등록
# scaler에서 fit(): 범위를 지정
# scaler에서 transform(): 지정된 범위를 이용하여 스케일링
# train data: fit() → transform() | fit_transform()
# test data: transform()

stdscaler.fit(X_train3)

,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True


In [21]:
# train 데이터 스케일링(변환) → 함수의 리턴 데이터를 이용하여 원본 데이터를 변화시키지 않고 유지

X_train_sc = stdscaler.transform(X_train3)

In [22]:
# test 데이터 스케일링

X_test_sc = stdscaler.transform(X_test3)

In [26]:
# 데이터들의 최솟값과 최댓값 평균, 표준편차를 출력하는 함수를 생성

def scaler_print(train, test):
    print(f"""
    Train Data:
        Min : {round(train.min(), 2)}
        Max : {round(train.max(), 2)}
        Mean: {round(train.mean(), 2)}
        Std : {round(train.std(), 2)}
""")

    print(f"""
    Test Data:
        Min : {round(test.min(), 2)}
        Max : {round(test.max(), 2)}
        Mean: {round(test.mean(), 2)}
        Std : {round(test.std(), 2)}
""")

In [27]:
scaler_print(X_train_sc, X_test_sc)


    Train Data:
        Min : -2.32
        Max : 2.96
        Mean: -0.0
        Std : 1.0


    Test Data:
        Min : -1.72
        Max : 2.08
        Mean: -0.04
        Std : 0.9



In [29]:
scaler_print(X_train3.values, X_test3.values)


    Train Data:
        Min : 0.1
        Max : 7.9
        Mean: 3.48
        Std : 1.99


    Test Data:
        Min : 0.2
        Max : 7.3
        Mean: 3.43
        Std : 1.93



#### Min-Max Scaler
- 정규화 방식으로 컬럼의 데이터들을 0과 1 사이의 값으로 스케일링하는 방식
- 최솟값 0, 최댓값 1
- 이상치에 매우 민감하므로 이상치에 대한 확인/정제 필수
- 일반적으로 분류 분석보다는 회귀 분석에서 사용

In [30]:
from sklearn.preprocessing import MinMaxScaler

mnscaler = MinMaxScaler()

X_train_sc = mnscaler.fit_transform(X_train3)
X_test_sc = mnscaler.transform(X_test3)

scaler_print(X_train_sc, X_test_sc)


    Train Data:
        Min : 0.0
        Max : 1.0
        Mean: 0.45
        Std : 0.27


    Test Data:
        Min : -0.02
        Max : 0.96
        Mean: 0.44
        Std : 0.25



#### Max Abs Scaler
- 최대절댓값과 0이 각각 1, 0이 되도록 스케일링을 하는 정규화 방식, 모든 값은 -1부터 1 사이로 표현
- 스케일링 대상의 데이터가 모두 양수라면 MinMaxScaler와 동일
- 이상치에 매우 민감
- 일반적으로는 분류 분석보다는 회귀 분석에서 사용

In [32]:
from sklearn.preprocessing import MaxAbsScaler
mascaler = MaxAbsScaler()

X_train_sc = mascaler.fit_transform(X_train3)
X_test_sc = mascaler.transform(X_test3)

scaler_print(X_train_sc, X_test_sc)


    Train Data:
        Min : 0.04
        Max : 1.0
        Mean: 0.62
        Std : 0.24


    Test Data:
        Min : 0.08
        Max : 0.96
        Mean: 0.61
        Std : 0.23



#### Robust Scaler

- 평균과 분산을 이용한 Standard 대신에 중앙값과 사분위수를 활용하는 방식
- 중앙값을 0으로 설정, IQR을 사용하여 이상치의 영향을 최소화
- quantile_range 매개변수(기본값 (25.0, 75.0)): 더 넓거나 좁은 범위의 값을 이상치로 판단하게 할 수 있다.

In [34]:
from sklearn.preprocessing import RobustScaler
ruscaler = RobustScaler()
ruscaler2 = RobustScaler(quantile_range=(20, 80))

In [35]:
X_train_sc = ruscaler.fit_transform(X_train3)
X_test_sc = ruscaler.transform(X_test3)
scaler_print(X_train_sc, X_test_sc)


    Train Data:
        Min : -2.0
        Max : 2.8
        Mean: -0.0
        Std : 0.67


    Test Data:
        Min : -1.4
        Max : 2.0
        Mean: -0.03
        Std : 0.59



In [36]:
X_train_sc = ruscaler2.fit_transform(X_train3)
X_train_sc = ruscaler2.transform(X_test3)
scaler_print(X_train_sc, X_test_sc)


    Train Data:
        Min : -1.0
        Max : 1.43
        Mean: -0.03
        Std : 0.47


    Test Data:
        Min : -1.4
        Max : 2.0
        Mean: -0.03
        Std : 0.59



In [38]:
pd.DataFrame(X_train_sc).head()

,0,1,2,3
0,0.9375,-0.142857,0.535714,0.277778
1,0.1875,-0.142857,0.127551,0.055556
2,0.3125,-0.285714,0.229592,0.111111
3,0.3125,0.428571,0.127551,0.166667
4,0.1875,0.000000,0.178571,0.277778


In [40]:
# ↑ 저대로 쓸 수는 없으니
# 스케일링이 된 데이터를 원본으로 복원

X_origin = ruscaler2.inverse_transform(X_train_sc)
pd.DataFrame(X_origin).head()

,0,1,2,3
0,7.3,2.9,6.3,1.8
1,6.1,2.9,4.7,1.4
2,6.3,2.8,5.1,1.5
3,6.3,3.3,4.7,1.6
4,6.1,3.0,4.9,1.8


In [41]:
X_train.head()

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm)
81,5.5,2.4,3.7,1.0
133,6.3,2.8,5.1,1.5
137,6.4,3.1,5.5,1.8
75,6.6,3.0,4.4,1.4
109,7.2,3.6,6.1,2.5
